# Agentic banking pipeline: from raw transactions to LLM-ready data

This project shows how a data pipeline can feed an LLM-based agent safely. The raw transactions are cleaned and validated, personal data is masked, and the results are turned into:

1. **Account-month briefs:** short text summaries that a language model can read and cite.
2. **A search index:** finds the right briefs for a question (TF-IDF, no external API needed).
3. **Tools:** typed functions with JSON schemas that an agent can call. The agent never sees raw customer IDs.
4. **An agent:** runs offline with a simple router, and optionally with the Claude API if you set `ANTHROPIC_API_KEY`.
5. **Guardrails:** checks that stop personal data from leaking into the LLM-ready layer.

The notebook needs only `pandas`, `numpy`, `scikit-learn`. The LLM step is optional.

In [ ]:
import os  # work with files and environment variables
import re  # regular expressions, used to pull account IDs and months from questions
import json  # read and write JSON, used for briefs and tool schemas
import hashlib  # create one-way hashes, used to mask customer IDs
import pandas as pd  # tables for the cleaning and aggregation steps
import numpy as np  # numeric helpers
from sklearn.feature_extraction.text import TfidfVectorizer  # turns text into numeric vectors for search
from sklearn.metrics.pairwise import cosine_similarity  # measures how similar a question is to each brief

In [ ]:
INPUT_FILE = 'bank_transactions_agentic.csv'  # the raw transaction feed
OUTPUT_DIR = 'output_agentic'  # where the LLM-ready files are written
SALT = 'change-this-salt-before-real-use'  # secret added before hashing customer IDs; keep it out of version control
MODEL = 'claude-haiku-5-5'  # Claude model used by the optional agent step
LARGE_THRESHOLD = 10000  # amounts at or above this are flagged for review
os.makedirs(OUTPUT_DIR, exist_ok=True)  # create the output folder if it does not exist

## 1. Ingest and validate
Read everything as text, then fix formatting and types. Rows that break a rule go to quarantine with a reason code.

In [ ]:
raw = pd.read_csv(INPUT_FILE, dtype=str)  # read every column as text so nothing is changed silently
df = raw.copy()  # keep the raw table unchanged and work on a copy
df['transaction_type'] = df['transaction_type'].str.strip().str.upper()  # 'debit ' becomes 'DEBIT'
df['currency'] = df['currency'].str.strip().str.upper()  # 'mur' becomes 'MUR'
df['status'] = df['status'].str.strip().str.upper()  # standardise status text
df['merchant_category'] = df['merchant_category'].fillna('').str.strip().replace('', 'UNKNOWN')  # blank category becomes 'UNKNOWN'
df['account_id'] = df['account_id'].str.strip().replace('', np.nan)  # blank account IDs become missing values
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')  # text to number; blank or invalid becomes NaN
df['balance_after'] = pd.to_numeric(df['balance_after'], errors='coerce')  # same conversion for balances
df['transaction_date'] = pd.to_datetime(df['transaction_date'], format='%Y-%m-%d', errors='coerce')  # impossible dates become NaT
content_cols = [c for c in df.columns if c != 'transaction_id']  # every field except the ID describes the money movement
df = df.drop_duplicates(subset=content_cols, keep='first')  # remove exact copies that have a different ID
print('Raw rows:', len(raw), '| after removing copies:', len(df))  # show the effect of de-duplication

In [ ]:
df['reason_codes'] = df.apply(lambda r: ','.join(code for code, bad in [  # build a list of reason codes for each row
    ('MISSING_ACCOUNT', pd.isna(r['account_id'])),  # no account, so the row cannot be linked
    ('MISSING_AMOUNT', pd.isna(r['amount'])),  # no money value, so it must not be guessed
    ('INVALID_DATE', pd.isna(r['transaction_date'])),  # date missing or impossible
    ('NEGATIVE_AMOUNT', (not pd.isna(r['amount'])) and r['amount'] < 0),  # sign convention unclear
    ('FAILED_STATUS', r['status'] != 'SUCCESS'),  # failed transactions did not move money
] if bad), axis=1)  # join the codes that apply, separated by commas
quarantine = df[df['reason_codes'] != ''].copy()  # rows with at least one failed rule
curated = df[df['reason_codes'] == ''].drop(columns='reason_codes').copy()  # rows that passed every rule
print('Clean:', len(curated), '| Quarantined:', len(quarantine))  # show the split
quarantine[['transaction_id', 'account_id', 'reason_codes']]  # display the quarantined rows and their codes

## 2. Curate and mask personal data
Customer IDs are replaced with a salted hash. Only the token goes into anything an LLM can see.

In [ ]:
def mask_customer(cust):  # turn a customer ID into a short, stable token
    digest = hashlib.sha256((SALT + str(cust)).encode('utf-8')).hexdigest()  # salted SHA-256 of the customer ID
    return 'cust_' + digest[:10]  # keep the first ten characters as a readable token
curated['customer_token'] = curated['customer_id'].apply(mask_customer)  # add the masked token for every clean row
curated['txn_month'] = curated['transaction_date'].dt.strftime('%Y-%m')  # month label used for grouping, such as 2026-08
curated['signed_amount'] = np.where(curated['transaction_type'] == 'CREDIT', curated['amount'], -curated['amount'])  # + money in, - money out
curated['is_large'] = curated['amount'] >= LARGE_THRESHOLD  # flag amounts for AML review
curated = curated.drop(columns='customer_id')  # remove the raw customer ID from the curated table
quarantine['customer_token'] = quarantine['customer_id'].apply(mask_customer)  # mask the quarantined rows the same way
quarantine = quarantine.drop(columns='customer_id')  # remove the raw customer ID from the quarantine table too
print(curated[['transaction_id', 'account_id', 'customer_token', 'txn_month', 'signed_amount', 'is_large']].head(3).to_string(index=False))  # preview the masked rows

## 3. Build account-month briefs
A brief is a short, factual paragraph for one account and one month. It is the unit that gets indexed, retrieved and cited.

In [ ]:
briefs = []  # list that will hold one brief per account and month
q_valid = quarantine.dropna(subset=['transaction_date']).copy()  # quarantined rows that have a usable month
q_valid['txn_month'] = q_valid['transaction_date'].dt.strftime('%Y-%m')  # month label for the quarantined rows
for (acc, month), g in curated.groupby(['account_id', 'txn_month']):  # one group per account and month
    ccy = g['currency'].mode().iloc[0]  # the account's currency (one per account in this data)
    ok = g[g['status'] == 'SUCCESS']  # only successful transactions move money
    money_in = ok.loc[ok['transaction_type'] == 'CREDIT', 'amount'].sum()  # total credited
    money_out = ok.loc[ok['transaction_type'] == 'DEBIT', 'amount'].sum()  # total debited
    debits = ok[ok['transaction_type'] == 'DEBIT']  # debit rows only
    largest = debits.loc[debits['amount'].idxmax()] if len(debits) else None  # the single largest debit
    failed = int((g['status'] != 'SUCCESS').sum())  # failed transactions in the month
    large = int(g['is_large'].sum())  # transactions flagged for review
    q_here = q_valid[(q_valid['account_id'] == acc) & (q_valid['txn_month'] == month)]  # quarantined rows in the same month
    q_text = ', '.join(sorted(set(q_here['reason_codes']))) or 'none'  # reason codes that appeared, or 'none'
    largest_text = f"Largest debit {largest['amount']:,.2f} {ccy} ({largest['merchant_category']})" if largest is not None else 'no debits'  # describe the largest debit
    text = (f"Account {acc}, {month}, currency {ccy}. Customer token {g['customer_token'].iloc[0]}. "  # opening sentence with the masked token
            f"Transactions: {len(g)} in total, of which {len(ok)} succeeded and {failed} failed. "  # counts
            f"Money in {money_in:,.2f} {ccy}; money out {money_out:,.2f} {ccy}; net flow {money_in - money_out:+,.2f} {ccy}. "  # totals in one currency only
            f"Transactions at or above {LARGE_THRESHOLD:,} flagged for review: {large}. "  # AML review count
            f"Quarantined rows this month: {len(q_here)} (reasons: {q_text}). {largest_text}.")  # quarantine and largest debit
    briefs.append({'doc_id': f'{acc}_{month}', 'account_id': acc, 'month': month, 'currency': ccy, 'text': text,  # keep one brief record, with its numbers
                   'net_flow': round(float(money_in - money_out), 2), 'large_count': large, 'failed_count': failed})  # keep the numbers too
briefs_df = pd.DataFrame(briefs)  # table of briefs, used by the tools
print('Briefs created:', len(briefs_df))  # confirm how many were written
print(briefs_df['text'].iloc[0])  # show one example brief

## 4. Search index
TF-IDF turns each brief into a vector of word weights. A question is turned into the same kind of vector, and the closest briefs are returned. This runs offline.

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))  # word and two-word weights, English stop words removed
brief_matrix = vectorizer.fit_transform(briefs_df['text'])  # one vector per brief
def search_briefs(query, k=3):  # return the k briefs most similar to the query
    qv = vectorizer.transform([query])  # turn the question into the same vector space
    scores = cosine_similarity(qv, brief_matrix).ravel()  # similarity between the question and every brief
    top = np.argsort(scores)[::-1][:int(k)]  # indices of the best matches, highest first
    return [{'doc_id': briefs_df.iloc[i]['doc_id'], 'score': round(float(scores[i]), 3), 'text': briefs_df.iloc[i]['text']} for i in top if scores[i] > 0]  # keep only real matches
search_briefs('failed transactions', k=2)  # try a search: briefs that mention failed transactions

## 5. Tools for the agent
Each tool is a plain Python function plus a JSON schema. The schema tells the model what the tool does and what arguments it takes. The tools only return masked data.

In [ ]:
def get_account_month(account_id: str, month: str) -> dict:  # return the brief numbers for one account and month
    hit = briefs_df[(briefs_df['account_id'] == account_id) & (briefs_df['month'] == month)]  # find the matching brief
    if hit.empty:  # nothing found
        return {'error': f'no brief for {account_id} in {month}'}  # tell the caller clearly
    row = hit.iloc[0]  # take the single matching row
    return {'account_id': row['account_id'], 'month': row['month'], 'currency': row['currency'],  # return the facts and the brief text
            'net_flow': row['net_flow'], 'failed_count': int(row['failed_count']), 'large_count': int(row['large_count']), 'text': row['text']}  # return the facts and the brief

def list_quarantine(reason_code: str = '') -> dict:  # list quarantined rows, optionally for one reason code
    rows = quarantine if not reason_code else quarantine[quarantine['reason_codes'].str.contains(reason_code)]  # filter by reason if given
    out = rows[['transaction_id', 'account_id', 'reason_codes']].head(20).to_dict(orient='records')  # at most 20 rows
    return {'count': int(len(rows)), 'rows': out}  # return the count and the rows

def get_large_transactions(min_amount: float = LARGE_THRESHOLD) -> dict:  # list clean transactions at or above an amount
    rows = curated[curated['amount'] >= float(min_amount)]  # filter clean rows by amount
    out = rows[['transaction_id', 'account_id', 'transaction_date', 'amount', 'currency', 'channel']].head(20).copy()  # at most 20 rows
    out['transaction_date'] = out['transaction_date'].dt.strftime('%Y-%m-%d')  # dates as plain text for JSON
    return {'count': int(len(rows)), 'rows': out.to_dict(orient='records')}  # return the count and the rows

def accounts_with_failures(min_failed: int = 1) -> dict:  # list account-months that had failed transactions
    hits = briefs_df[briefs_df['failed_count'] >= int(min_failed)]  # filter the briefs by their failed count
    out = hits[['account_id', 'month', 'failed_count']].sort_values(['failed_count', 'account_id'], ascending=[False, True]).to_dict(orient='records')  # most failures first
    return {'count': int(len(hits)), 'rows': out}  # return the count and the rows

def search_briefs_tool(query: str, k: int = 3) -> dict:  # wrap the search function so it returns a dictionary
    return {'results': search_briefs(query, k)}  # the agent receives a dictionary, not a Python list

TOOL_FUNCS = {'get_account_month': get_account_month, 'list_quarantine': list_quarantine,  # map tool names to functions
              'get_large_transactions': get_large_transactions, 'accounts_with_failures': accounts_with_failures,  # more tool names mapped to functions
              'search_briefs': search_briefs_tool}  # last tool name mapped to its function

TOOL_SPECS = [  # JSON schemas that describe each tool to the model
    {'name': 'get_account_month', 'description': 'Get the monthly summary for one account. Use the account ID (for example ACC2001) and month (YYYY-MM).',  # schema for the account summary tool
     'input_schema': {'type': 'object', 'properties': {'account_id': {'type': 'string'}, 'month': {'type': 'string', 'description': 'YYYY-MM'}}, 'required': ['account_id', 'month']}},  # the tool's arguments
    {'name': 'list_quarantine', 'description': 'List rows that failed validation. Optionally filter by reason code such as INVALID_DATE, FAILED_STATUS or MISSING_AMOUNT.',  # schema for the quarantine tool
     'input_schema': {'type': 'object', 'properties': {'reason_code': {'type': 'string'}}, 'required': []}},  # optional reason-code argument
    {'name': 'get_large_transactions', 'description': 'List clean transactions at or above an amount, for AML review.',  # schema for the large transaction tool
     'input_schema': {'type': 'object', 'properties': {'min_amount': {'type': 'number'}}, 'required': []}},  # optional minimum amount
    {'name': 'accounts_with_failures', 'description': 'List account-months with at least min_failed failed transactions, most failures first. Use this for counting questions about failures.',  # schema for the failure tool
     'input_schema': {'type': 'object', 'properties': {'min_failed': {'type': 'integer'}}, 'required': []}},  # optional minimum failure count
    {'name': 'search_briefs', 'description': 'Search the account-month briefs by meaning and keywords. Use for descriptive questions, not for exact counts.',  # schema for the search tool
     'input_schema': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'k': {'type': 'integer'}}, 'required': ['query']}},  # query text and number of results
]  # end of the list

def run_tool(name: str, args: dict) -> dict:  # run one tool by name with its arguments
    if name not in TOOL_FUNCS:  # unknown tool names are refused
        return {'error': f'unknown tool {name}'}  # say which tool was unknown
    return TOOL_FUNCS[name](**args)  # call the function with the arguments the model supplied

print(json.dumps(run_tool('get_account_month', {'account_id': 'ACC2001', 'month': '2026-08'}), indent=1)[:600])  # test one tool call

## 6. Offline agent
Without an API key, a simple router picks the tool from the question. It shows the agent loop without needing any external service.

In [ ]:
def route(question: str) -> dict:  # decide which tool to call, using simple rules
    q = question.lower()  # lower case so matching is simple
    acc = re.search(r'acc\d{4}', q)  # look for an account ID such as ACC2001
    month = re.search(r'\d{4}-\d{2}', q)  # look for a month such as 2026-08
    code_match = re.search(r'[A-Z_]{6,}', question)  # look for a reason code in upper case
    if acc and month:  # an account and a month: summary tool
        return {'tool': 'get_account_month', 'args': {'account_id': acc.group(0).upper(), 'month': month.group(0)}}  # choose the account-month tool
    if 'quarantin' in q or 'reject' in q:  # questions about held-back rows
        return {'tool': 'list_quarantine', 'args': {'reason_code': code_match.group(0) if code_match else ''}}  # choose the quarantine tool
    if 'large' in q or 'aml' in q or 'review' in q:  # questions about large transactions
        return {'tool': 'get_large_transactions', 'args': {}}  # choose the large-transaction tool
    if 'fail' in q:  # questions about failed transactions
        return {'tool': 'accounts_with_failures', 'args': {}}  # choose the failure tool
    return {'tool': 'search_briefs', 'args': {'query': question, 'k': 3}}  # anything else goes to search

questions = [  # four example questions
    'Show the summary for ACC2001 in 2026-08',  # question about one account and month
    'Which transactions were quarantined for INVALID_DATE?',  # question about quarantined rows
    'Which large transactions need AML review?',  # question about large transactions
    'Which accounts had failed transactions?',  # question about failed transactions
]  # end of the list
for q in questions:  # answer each question in turn
    step = route(q)  # choose the tool
    result = run_tool(step['tool'], step['args'])  # run it
    print('Q:', q)  # show the question
    print('  tool:', step['tool'], step['args'])  # show the tool and arguments
    print('  result:', json.dumps(result, default=str)[:300], '...')  # show the first part of the result

## 7. Optional: Claude as the agent
If `ANTHROPIC_API_KEY` is set, Claude receives the tool schemas and decides which tools to call. The loop sends each tool result back until Claude gives a final answer.

In [ ]:
SYSTEM = ('You answer questions about banking transactions. Use the tools for every figure, and never invent numbers. '  # system instructions for the model
          'Refer to customers only by their token. If a tool returns an error, say so.')  # rules for the model
def ask_claude(question: str, max_steps: int = 5) -> str:  # run the agent loop with Claude
    import anthropic  # import here so the notebook still runs without the package
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    messages = [{'role': 'user', 'content': question}]  # start the conversation with the question
    for _ in range(max_steps):  # stop after a fixed number of steps
        resp = client.messages.create(model=MODEL, max_tokens=800, system=SYSTEM, tools=TOOL_SPECS, messages=messages)  # ask the model
        messages.append({'role': 'assistant', 'content': resp.content})  # keep the model's reply in the history
        if resp.stop_reason != 'tool_use':  # no more tools requested
            return ''.join(b.text for b in resp.content if b.type == 'text')  # return the final text
        results = []  # tool results for this round
        for block in resp.content:  # each block may be a tool request
            if block.type == 'tool_use':  # only tool requests need an answer
                out = run_tool(block.name, block.input)  # run the tool the model asked for
                results.append({'type': 'tool_result', 'tool_use_id': block.id, 'content': json.dumps(out, default=str)})  # send the result back
        messages.append({'role': 'user', 'content': results})  # add the results to the conversation
    return 'Stopped after the maximum number of steps.'  # safety stop

if os.environ.get('ANTHROPIC_API_KEY'):  # run only if a key is set
    print(ask_claude('Which accounts had the most failed transactions in 2026-08, and how many large transactions did they have?'))  # example question
else:  # when no key is set
    print('ANTHROPIC_API_KEY is not set, so the Claude step is skipped. The offline agent above still works.')  # clear message when skipped

## 8. Guardrails
Before anything is written for an LLM, these checks confirm that no raw customer ID appears, every brief has a valid month, and all tool schemas are well-formed.

In [ ]:
raw_customers = set(raw['customer_id'].dropna().unique())  # every raw customer ID from the feed
all_text = ' '.join(briefs_df['text']) + json.dumps(TOOL_SPECS)  # everything an LLM could read
all_text += curated.to_csv(index=False) + quarantine.to_csv(index=False)  # include the tables the tools read from
leaked = [c for c in raw_customers if c in all_text]  # any raw customer ID found in that text
assert not leaked, f'Raw customer IDs leaked into LLM-ready data: {leaked}'  # stop the run if personal data leaked
assert briefs_df['month'].str.match(r'^\d{4}-\d{2}$').all(), 'A brief has a malformed month'  # every month must look like YYYY-MM
assert all(set(s) >= {'name', 'description', 'input_schema'} for s in TOOL_SPECS), 'A tool schema is incomplete'  # every tool has a name, description and schema
print('Guardrails passed: no raw customer IDs in LLM-ready data')  # confirm the checks passed

## 9. Save the LLM-ready outputs
Briefs are written as JSON Lines, one brief per line, which most retrieval tools can load directly.

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'briefs.jsonl'), 'w', encoding='utf-8') as f:  # open the briefs file for writing
    for rec in briefs:  # one brief per line
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')  # write the record as one JSON object
with open(os.path.join(OUTPUT_DIR, 'tool_specs.json'), 'w', encoding='utf-8') as f:  # save the tool schemas
    json.dump(TOOL_SPECS, f, indent=2)  # pretty-printed for reading
curated.to_csv(os.path.join(OUTPUT_DIR, 'curated_masked.csv'), index=False)  # masked clean data for analysts
quarantine.to_csv(os.path.join(OUTPUT_DIR, 'quarantine.csv'), index=False)  # quarantined rows with reason codes
print('Saved to', OUTPUT_DIR, '->', sorted(os.listdir(OUTPUT_DIR)))  # list what was written